# Beyond the GPU: TPUs, LPUs and What Specialization Buys

The full code of [Beyond the GPU: TPUs, LPUs and What Specialization Buys](https://residual-stream.lvtan71.workers.dev/posts/beyond-the-gpu-tpus-and-lpus/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/beyond-the-gpu-tpus-and-lpus) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/beyond-the-gpu-tpus-and-lpus.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 1 min end to end on the RTX 3090 machine that saved the outputs below.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions change timings and the last digits of some floating-point results.
Sampled text and small training runs can move further; where that changes a
number the post discusses, the note above says so.

**Outputs saved from a run on NVIDIA GeForce RTX 3090 on 2026-10-18** (torch 2.14.1+cu130, 0 min; every cell ran). Run the notebook to replace them with your own.

In [1]:
%pip install -q "numpy==2.*"

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "beyond-the-gpu-tpus-and-lpus"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

Note: you may need to restart the kernel to use updated packages.
working in residual-stream-code/posts/beyond-the-gpu-tpus-and-lpus


## `systolic.py`

A 3×3 output-stationary systolic array computing C = A @ B cycle by cycle: how many cells work each cycle, a check against A @ B, and how often each value read from memory is reused for n = 3, 128, 256.

In [2]:
!{sys.executable} systolic.py

cycle 0: 1 of 9 cells multiplying
cycle 1: 3 of 9 cells multiplying
cycle 2: 6 of 9 cells multiplying
cycle 3: 7 of 9 cells multiplying
cycle 4: 6 of 9 cells multiplying
cycle 5: 3 of 9 cells multiplying
cycle 6: 1 of 9 cells multiplying

result equals A @ B: True   (27 multiply-adds in 7 cycles)
[[ 18  24  30]
 [ 54  69  84]
 [ 90 114 138]]
n =   3:         27 multiply-adds from      18 values read: each value used   3 times
n = 128:  2,097,152 multiply-adds from  32,768 values read: each value used 128 times
n = 256: 16,777,216 multiply-adds from 131,072 values read: each value used 256 times


The post printed:

```text
cycle 0: 1 of 9 cells multiplying
cycle 1: 3 of 9 cells multiplying
cycle 2: 6 of 9 cells multiplying
cycle 3: 7 of 9 cells multiplying
cycle 4: 6 of 9 cells multiplying
cycle 5: 3 of 9 cells multiplying
cycle 6: 1 of 9 cells multiplying

result equals A @ B: True   (27 multiply-adds in 7 cycles)
[[ 18  24  30]
 [ 54  69  84]
 [ 90 114 138]]
n =   3:         27 multiply-adds from      18 values read: each value used   3 times
n = 128:  2,097,152 multiply-adds from  32,768 values read: each value used 128 times
n = 256: 16,777,216 multiply-adds from 131,072 values read: each value used 256 times
```

## `sram.py`

How many chips it takes to hold a 70B model's weights at 16 and 8 bits on a GroqChip, an H100, a TPU v6e and a TPU7x, and the bandwidth-or-compute bound on one stream's tokens per second (vendor specs).

In [3]:
!{sys.executable} sram.py

70B model, 16-bit weights: 140 GB
  chip               chips to hold it  tokens/s bound  on 8 chips
  GroqChip (SRAM)                 609             571           -
  H100 SXM (HBM)                    2              48         191
  TPU v6e (HBM)                     5              58          94
  TPU7x (HBM)                       1              53         422

70B model, 8-bit weights: 70 GB
  chip               chips to hold it  tokens/s bound  on 8 chips
  GroqChip (SRAM)                 305            1143           -
  H100 SXM (HBM)                    1              48         383
  TPU v6e (HBM)                     3              70         187
  TPU7x (HBM)                       1             105         843



The post printed:

```text
70B model, 16-bit weights: 140 GB
  chip               chips to hold it  tokens/s bound  on 8 chips
  GroqChip (SRAM)                 609             571           -
  H100 SXM (HBM)                    2              48         191
  TPU v6e (HBM)                     5              58          94
  TPU7x (HBM)                       1              53         422

70B model, 8-bit weights: 70 GB
  chip               chips to hold it  tokens/s bound  on 8 chips
  GroqChip (SRAM)                 305            1143           -
  H100 SXM (HBM)                    1              48         383
  TPU v6e (HBM)                     3              70         187
  TPU7x (HBM)                       1             105         843
```